# Setup

In [2]:
import os
import json
import time
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List

os.environ["GOOGLE_CLOUD_PROJECT"] = "project-038ccd57-3d62-4aac-8b5"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

client = genai.Client()

# Utils & Prompt

In [3]:
def grid_to_markdown(grid):
    rows = []
    rows.append("| " + " | ".join([""] * len(grid[0])) + " |")
    rows.append("|" + "|".join(["---"] * len(grid[0])) + "|")
    for row in grid:
        rows.append("| " + " | ".join(str(cell) for cell in row) + " |")
    return "\n".join(rows)

with open('killer_sudoku_cheat_sheet.md', 'r', encoding='utf-8') as file:
    cheat_sheet = file.read()

prompt = """
You are an expert puzzle solver. Your task is to solve a 6x6 Killer Sudoku grid based on the rules, strategies, and puzzle data provided below.

1. The Grid & Core Rules:
- The grid is 6x6, containing 6 rows and 6 columns.
- Notation: "r1c1" indicates the cell at row 1 and column 1.
- The grid consists of 6 rectangular 2x3 subgrids (2 rows tall, 3 columns wide).
- Standard Sudoku Rule: Each row, each column, and each 2x3 subgrid must contain the numbers 1 through 6 exactly once.
- Killer Sudoku Rule: The grid is divided into "Cages" (contiguous groups of cells). The sum of the numbers in each cage must perfectly match its provided target sum.
- Non-Repeating Rule: Numbers cannot repeat within a single cage.

2. Mathematical Facts & Solving Strategies:
- Sum to 21: Since the numbers 1 through 6 add up to 21, every completely filled row, column, and 2x3 subgrid must sum exactly to 21.
- A 6-cell cage must also have a target sum of 21.
- Deduction Tip: For any subset of cells (row, column, or subgrid), if all but one cell are filled, the final cell's value must be 21 minus the sum of the known cells.

3. Reference Material: The Cheat-Sheet
Important Interpretation Rules for the cheat-sheet:
- Combinations are written as sequences of individual digits without separators.
- Each digit in the sequence represents one distinct number in the combination.
- Example: "12" means the combination {{1, 2}}. "135" means the combination {{1, 3, 5}}.
Here is your cheat-sheet for cage combinations:
{cheat_sheet}

4. The Puzzle Instance:
Initial State:
{puzzle}

Cages:
{cages}

5. Output Requirement:
Return ONLY the final solved 6x6 grid. Do not include any explanation or extra text.
"""

class SudokuResult(BaseModel):
    board: List[List[str]] = Field(description="The solved 6x6 Sudoku grid as a 2D list of strings.")

# Evaluation Loop (2 models, 7 maps)

In [4]:
models_LLM = ["gemini-3.5-flash", "gemini-3.5-flash-lite"]
puzzle_files = [f"dataset/puzzle_{i:02d}.json" for i in range(1, 6)]

os.makedirs("outputs/gemini-3.5/single-prompt/flash", exist_ok=True)
os.makedirs("outputs/gemini-3.5/single-prompt/lite", exist_ok=True)

def validate_board(board, puzzle):
    # Compare the predicted board with the solution
    solution = puzzle['solution']
    try:
        for r in range(6):
            for c in range(6):
                if int(board[r][c]) != solution[r][c]:
                    return False
        return True
    except:
        return False

for model in models_LLM:
    print(f"\nEvaluating model: {model}")
    model_dir = "lite" if "lite" in model else "flash"
    
    for p_file in puzzle_files:
        print(f"  Running {p_file}...")
        with open(p_file, 'r', encoding='utf-8') as f:
            puzzle = json.load(f)
            
        # Transform cages
        cages_text = []
        for cage in puzzle['cages']:
            cells = cage['cells']
            cage_sum = cage['sum']
            cell_strs = [f"r{r+1}c{c+1}" for r, c in cells]
            cages_text.append("- " + " + ".join(cell_strs) + f" = {cage_sum}")
            
        markdown_table = grid_to_markdown(puzzle['puzzle'])
        
        if model == "gemini-3.1-flash-lite":
            config = types.GenerateContentConfig(
                temperature=0,
                response_json_schema=SudokuResult.model_json_schema(),
                response_mime_type="application/json",
                thinking_config=types.ThinkingConfig(
                    thinking_level=types.ThinkingLevel.MEDIUM
                )
            )
        else:
            config = types.GenerateContentConfig(
                temperature=0,
                response_json_schema=SudokuResult.model_json_schema(),
                response_mime_type="application/json"
            )
        
        start_time = time.time()
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt.format(puzzle=markdown_table, cages="\n".join(cages_text), cheat_sheet=cheat_sheet), 
                config=config
            )
            result = SudokuResult.model_validate_json(response.text)
            is_correct = validate_board(result.board, puzzle)
        except Exception as e:
            print(f"    Error: {e}")
            result = None
            is_correct = False

        elapsed = time.time() - start_time
        
        # Save output
        log_data = {
            "puzzle_id": puzzle['id'],
            "difficulty": puzzle['difficulty'],
            "model": model,
            "status": "Success" if is_correct else "Failed",
            "time_seconds": elapsed,
            "prediction": result.board if result else None
        }
        
        out_file = f"outputs/gemini-3.5/single-prompt/{model_dir}/result_puzzle_{puzzle['id']:02d}.json"
        with open(out_file, 'w', encoding='utf-8') as f:
            json.dump(log_data, f, ensure_ascii=False, indent=2)
            
        print(f"    Status: {'Success' if is_correct else 'Failed'}, Time: {elapsed:.2f}s")
        time.sleep(2) # Slight delay to respect rate limits


Evaluating model: gemini-3.5-flash
  Running dataset/puzzle_01.json...
    Status: Success, Time: 30.84s
  Running dataset/puzzle_02.json...
    Status: Success, Time: 23.98s
  Running dataset/puzzle_03.json...
    Status: Success, Time: 22.37s
  Running dataset/puzzle_04.json...
    Status: Success, Time: 26.84s
  Running dataset/puzzle_05.json...
    Status: Success, Time: 33.83s
